In [52]:
# from google.colab import drive
# drive.mount('/content/drive')

In [53]:
import math
import os
from typing import Any

import flax.nnx as nnx
import grain.python as grain
import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp

from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, PreTrainedTokenizer
from grain.checkpoint import CheckpointSave as GrainCheckpointSave, CheckpointRestore as GrainCheckpointRestore

In [54]:
def chunked_forward_optimized(
    query: jax.Array,
    key: jax.Array,
    value: jax.Array,
    beta: jax.Array,
    gamma: jax.Array,
    delta: jax.Array,
    chunk_size: int
) -> jax.Array:
    batch_size, seq_len, query_dim = query.shape
    value_dim = value.shape[-1]
    num_chunks = seq_len // chunk_size

    # 1. Reshape and transpose inputs to (num_chunks, batch, chunk_size, dim)
    # Transposing allows jax.lax.scan to iterate over the chunks (axis 0).
    def prepare_scan_input(x: jax.Array) -> jax.Array:
        x_reshaped = x.reshape(batch_size, num_chunks, chunk_size, -1)
        return x_reshaped.swapaxes(0, 1)

    q_scan = prepare_scan_input(query)
    k_scan = prepare_scan_input(key)
    v_scan = prepare_scan_input(value)
    b_scan = prepare_scan_input(beta)
    w_scan = prepare_scan_input(gamma)
    d_scan = prepare_scan_input(delta)

    I = jnp.eye(query_dim, dtype=query.dtype)

    # 2. Define the associative combination operator
    def combine(state1: tuple[jax.Array, jax.Array], state2: tuple[jax.Array, jax.Array]):
        A1, B1 = state1
        A2, B2 = state2

        # Batched matrix multiplications natively broadcast over the batch dimension
        A_out = A2 @ A1
        B_out = A2 @ B1 + B2
        return A_out, B_out

    # 3. Define the step function for the outer chunk scan
    def chunk_step(S_prev: jax.Array, xs: tuple):
        q_c, k_c, v_c, b_c, w_c, d_c = xs
        
        # Vectorized calculation of A_c and B_c for the entire chunk
        # outer_k shape: (batch, chunk_size, query_dim, query_dim)
        outer_k = jnp.einsum('bci,bcj->bcij', k_c, b_c * k_c)
        
        # d_c is expanded to broadcast across the columns of the matrix
        A_c = (I - outer_k) * jnp.expand_dims(d_c, axis=-2)
        B_c = jnp.einsum('bci,bcj->bcij', k_c, w_c * v_c)

        # Transpose chunk dimension to axis 0 for associative_scan
        A_c_t = A_c.swapaxes(0, 1)
        B_c_t = B_c.swapaxes(0, 1)

        # 4. Perform the associative scan to resolve the prefix matrices
        A_cum_t, B_cum_t = jax.lax.associative_scan(combine, (A_c_t, B_c_t))

        # Transpose back to (batch, chunk_size, ...)
        A_cum = A_cum_t.swapaxes(0, 1)
        B_cum = B_cum_t.swapaxes(0, 1)

        # 5. Compute the hidden states and outputs for all tokens in the chunk
        # S_all shape: (batch, chunk_size, query_dim, value_dim)
        S_all = jnp.einsum('bcij,bjk->bcik', A_cum, S_prev) + B_cum
        
        # o_c shape: (batch, chunk_size, value_dim)
        o_c = jnp.einsum('bci,bcij->bcj', q_c, S_all)

        # The state carried to the next chunk is the state of the final token
        S_next = S_all[:, -1, :]

        return S_next, o_c

    # Initial hidden state
    S_init = jnp.zeros((batch_size, query_dim, value_dim), dtype=query.dtype)

    # 6. Execute the compiled loop over the chunks
    _, o_scanned = jax.lax.scan(
        chunk_step,
        S_init,
        (q_scan, k_scan, v_scan, b_scan, w_scan, d_scan)
    )

    # Restore the original sequence dimension layout
    o_out = o_scanned.swapaxes(0, 1)
    return o_out.reshape(batch_size, num_chunks * chunk_size, value_dim)

# --- 1. The Gated DeltaNet Layer ---
class GatedDeltaNetLayer(nnx.Module):
    def __init__(self, dim: int, num_heads: int, chunk_size: int, rngs: nnx.Rngs):
        assert dim % num_heads == 0, "Dimension must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.chunk_size = chunk_size

        # Content Projections
        self.q_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.k_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.v_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

        # Gate Projections (Erase, Write, Decay)
        self.beta_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.gamma_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.delta_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

        # Output Projection
        self.o_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        batch_size, seq_len, dim = x.shape

        # 1. Project inputs
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # 2. Project and activate gates
        # Beta (Erase) and Gamma (Write) are typically sigmoid or swish
        beta = jax.nn.sigmoid(self.beta_proj(x))
        gamma = jax.nn.sigmoid(self.gamma_proj(x))
        # Delta (Decay) MUST be strictly (0, 1) to ensure recurrent stability
        delta = jax.nn.sigmoid(self.delta_proj(x))

        # 3. Reshape for Multi-Head: (B, L, H, D_h)
        def reshape_to_heads(tensor):
            return tensor.reshape(batch_size, seq_len, self.num_heads, self.head_dim)

        q, k, v = map(reshape_to_heads, (q, k, v))
        
        # Unit-norm keys to prevent explosive values in I - outer_k
        k = k / jnp.maximum(jnp.linalg.norm(k, axis=-1, keepdims=True), 1e-6)

        beta, gamma, delta = map(reshape_to_heads, (beta, gamma, delta))

        # 4. Flatten Batch and Heads to reuse your exact chunked_forward function
        # Shape becomes: (B * H, L, D_h)
        def flatten_batch_heads(tensor):
            # Transpose to (B, H, L, D_h) then reshape to (B*H, L, D_h)
            tensor = jnp.swapaxes(tensor, 1, 2) 
            return tensor.reshape(batch_size * self.num_heads, seq_len, self.head_dim)

        q_flat, k_flat, v_flat = map(flatten_batch_heads, (q, k, v))
        b_flat, g_flat, d_flat = map(flatten_batch_heads, (beta, gamma, delta))

        # 5. Call your optimized recurrence function!
        # (Make sure chunked_forward_optimized is in scope)
        o_flat = chunked_forward_optimized(
            q_flat, k_flat, v_flat, 
            b_flat, g_flat, d_flat, 
            self.chunk_size
        )

        # 6. Unflatten back to Multi-Head: (B, H, L, D_h)
        o = o_flat.reshape(batch_size, self.num_heads, seq_len, self.head_dim)
        
        # 7. Recombine heads: (B, L, H, D_h) -> (B, L, Dim)
        o = jnp.swapaxes(o, 1, 2).reshape(batch_size, seq_len, dim)

        # 8. Final output projection
        return self.o_proj(o)

# --- 2. The Model Block (Attention + MLP) ---
class GatedDeltaNetBlock(nnx.Module):
    def __init__(self, dim: int, num_heads: int, mlp_dim: int, chunk_size: int, rngs: nnx.Rngs):
        # Normalization
        self.norm1 = nnx.RMSNorm(dim, rngs=rngs)
        self.norm2 = nnx.RMSNorm(dim, rngs=rngs)

        # Core DeltaNet Layer
        self.attn = GatedDeltaNetLayer(dim, num_heads, chunk_size, rngs=rngs)

        # Standard Feed-Forward Network (GLU variant is standard for DeltaNet)
        self.mlp_w1 = nnx.Linear(dim, mlp_dim, use_bias=False, rngs=rngs)
        self.mlp_w2 = nnx.Linear(dim, mlp_dim, use_bias=False, rngs=rngs)
        self.mlp_w3 = nnx.Linear(mlp_dim, dim, use_bias=False, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        # Pre-norm architecture
        # 1. DeltaNet Sub-layer
        x = x + self.attn(self.norm1(x))
        
        # 2. MLP Sub-layer (SwiGLU)
        h = self.norm2(x)
        mlp_out = self.mlp_w3(jax.nn.silu(self.mlp_w1(h)) * self.mlp_w2(h))
        x = x + mlp_out
        
        return x

# --- 3. The Full Language Model ---
class GatedDeltaNet(nnx.Module):
    def __init__(self, vocab_size: int, num_layers: int, dim: int, num_heads: int, chunk_size: int, rngs: nnx.Rngs):
        self.embed = nnx.Embed(vocab_size, dim, rngs=rngs)
        
        # Stack multiple blocks
        self.blocks = nnx.Sequential(
            *[
                GatedDeltaNetBlock(dim, num_heads, mlp_dim=dim * 4, chunk_size=chunk_size, rngs=rngs) 
                for _ in range(num_layers)
            ]
        )
        
        self.norm_f = nnx.RMSNorm(dim, rngs=rngs)
        self.lm_head = nnx.Linear(dim, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, input_ids: jax.Array) -> jax.Array:
        x = self.embed(input_ids)
        
        x = self.blocks(x)
            
        x = self.norm_f(x)
        return self.lm_head(x)

In [55]:
# --- Wrap Hugging Face Dataset in a Grain Data Source ---
class HuggingFaceDataSource(grain.RandomAccessDataSource):
    """
    A Grain wrapper for Hugging Face datasets.
    Because HF relies on Apache Arrow under the hood, random lookups are incredibly fast.
    """
    def __init__(self, hf_ds: Dataset) -> None:
        self.hf_ds = hf_ds

    def __len__(self) -> int:
        return len(self.hf_ds)

    def __getitem__(self, index: int) -> dict[str, Any]:
        # HF natively returns a dictionary for the row (e.g., {'text': 'Some string'})
        return self.hf_ds[index]

In [56]:
class TokenizerAndShift(grain.MapTransform):
    def __init__(self, tokenizer: PreTrainedTokenizer, max_length: int = 128) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def map(self, element: dict[str, Any]) -> dict[str, Any]:
        encoded: dict[str, list[int]] = self.tokenizer(
            element["text"],
            truncation=True,
            max_length=self.max_length + 1,  # +1 for the shift
            padding="max_length",
            return_tensors=None,
        )

        tokens: list[int] = encoded["input_ids"]

        new_element = {
            "inputs": tokens[:-1],
            "targets": tokens[1:],
            "attention_mask": encoded["attention_mask"][:-1],
        }

        return new_element

In [57]:
class ConvertToJaxArrays(grain.MapTransform):
    def map(self, element: dict[str, Any]) -> dict[str, Any]:
        for key in ["inputs", "targets", "attention_mask"]:
            element[key] = jnp.array(np.array(element[key]))
        return element

In [58]:
class FilterEmptyLines(grain.FilterTransform):
    def filter(self, element: dict[str, Any]) -> bool:
        return len(element["text"].strip()) > 0

In [59]:
def build_dataloader(
    dataset: Dataset,
    tokenizer: PreTrainedTokenizer,
    batch_size: int = 8,
    max_length: int = 128,
) -> grain.DataLoader:
    source = HuggingFaceDataSource(dataset)

    sampler = grain.IndexSampler(
        num_records=len(source),
        num_epochs=1,
        shard_options=grain.ShardOptions(
            shard_index=0, shard_count=1, drop_remainder=True
        ),
        shuffle=True,
        seed=42,
    )

    loader = grain.DataLoader(
        data_source=source,
        sampler=sampler,
        operations=[
            FilterEmptyLines(),
            TokenizerAndShift(tokenizer, max_length=max_length),
            ConvertToJaxArrays(),
            grain.Batch(batch_size=batch_size, drop_remainder=True),
        ],
        worker_count=0,
    )

    return loader

In [60]:
def loss_fn(model: nnx.Module, batch: dict[str, jax.Array]) -> jax.Array:
    logits = model(batch["inputs"])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch["targets"]
    ).mean()

    return loss

In [61]:
@nnx.jit
def train_step(
    model: nnx.Module, optimizer: nnx.Optimizer, batch: dict[str, jax.Array]
) -> jax.Array:
    grad_fn = nnx.value_and_grad(loss_fn)
    loss, grads = grad_fn(model, batch)

    optimizer.update(model, grads)
    return loss

In [62]:
@nnx.jit
def eval_step(model: nnx.Module, batch: dict[str, jax.Array]) -> jax.Array:
    logits = model(batch["inputs"])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch["targets"]
    ).mean()

    return loss

In [63]:
def save_checkpoint(
    mngr: ocp.CheckpointManager,
    step: int,
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    rngs: nnx.Rngs,
    data_iterator
) -> None:
    """Bundles model weights, optimizer momentum, and Grain iterator state to disk."""
    print(f"Saving checkpoint at step {step}...")

    # 1. Get the NNX state (Model + Optimizer)
    # _, nnx_state = nnx.split((model, optimizer))

    # 2. Get the Grain iterator state
    # grain_state = data_iterator.get_state()

    # 3. Create a unified PyTree dictionary
    unified_state = {
        "model": ocp.args.StandardSave(model),
        "optimizer": ocp.args.StandardSave(optimizer),
        "rngs": ocp.args.StandardSave(rngs),
    }

    # 4. Save the unified state
    mngr.save(
        step,
        args=ocp.args.Composite(**unified_state)
    )

    # Block until save is complete to ensure safety
    mngr.wait_until_finished()
    print("Save complete!")

In [64]:
def restore_checkpoint(
    mngr: ocp.CheckpointManager,
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    rngs: nnx.Rngs,
    data_iterator
) -> int:
    """Restores the unified state and injects it back into the objects."""

    latest_step = mngr.latest_step()
    if latest_step is None:
        print("No existing checkpoints found. Starting from scratch (Step 0).")
        return 0

    print(f"Found checkpoint at step {latest_step}. Restoring...")

    # 1. Create the abstract template for NNX
    # _, abstract_nnx_state = nnx.split((model, optimizer))

    # 2. Create the abstract template for Grain
    # (We can just use its current empty state as the template)
    # abstract_grain_state = data_iterator.get_state()

    # 3. Assemble the abstract unified state
    abstract_unified_state = {
        "model": ocp.args.StandardRestore(model),
        "optimizer": ocp.args.StandardRestore(optimizer),
        "rngs": ocp.args.StandardRestore(rngs),
    }

    # 4. Restore from disk
    restored_state = mngr.restore(
        latest_step,
        args=ocp.args.Composite(**abstract_unified_state)
    )

    # 5. Inject the restored states back into their respective objects
    # nnx.update((model, optimizer), restored_state["nnx"])
    # data_iterator.set_state(restored_state["grain"])

    print("Restore complete! Model, Optimizer, and DataLoader are synchronized.")
    return latest_step

In [65]:
@nnx.jit
def predict_next_token(model: nnx.Module, input_ids: jax.Array) -> jax.Array:
    """Runs a single forward pass to predict the next word."""

    # 1. Get the raw scores (logits) for every token in the vocabulary
    logits = model(input_ids)

    # 2. Isolate the predictions for the very last token in our sequence
    # Shape goes from (batch, seq_len, vocab_size) -> (vocab_size,)
    next_token_logits = logits[0, -1, :]

    # 3. Greedy Decoding: Simply pick the token with the highest probability score
    next_token = jnp.argmax(next_token_logits)

    return next_token

In [66]:
def interactive_chat(model: nnx.Module, tokenizer: PreTrainedTokenizer, max_new_tokens: int = 100):
    """Starts an infinite loop for user interaction."""

    print("\n" + "="*50)
    print("🤖 Massive LLM is online and ready!")
    print("Type 'quit' or 'exit' to shut down the server.")
    print("="*50 + "\n")

    # The Infinite Loop
    while True:
        # 1. Get User Input
        user_text = input("You: ")

        # 2. Check for exit commands
        if user_text.strip().lower() in ['quit', 'exit']:
            print("Shutting down the model. Goodbye!")
            break

        # Skip empty inputs
        if not user_text.strip():
            continue

        # 3. Tokenize the input into a standard NumPy array
        # We add batch dimension manually so shape is (1, seq_len)
        encoded = tokenizer(user_text, return_tensors="np")
        input_ids = encoded["input_ids"]

        print("Model: ", end="", flush=True)

        # 4. The Autoregressive Generation Loop
        for _ in range(max_new_tokens):

            # A. Predict the next token
            next_token_array = predict_next_token(model, input_ids)

            # Convert the JAX array back to a standard Python integer
            next_token_id = next_token_array.item()

            # B. Check for the End-Of-Sequence (EOS) token
            # If the model decides it is done talking, break the generation loop
            if next_token_id == tokenizer.eos_token_id:
                break

            # C. Decode the single integer back into a readable word
            word = tokenizer.decode([next_token_id])

            # Print the word immediately (flush=True forces the terminal to update)
            print(word, end="", flush=True)

            # D. Append the new token to our sequence so the model can read it
            # on the next loop iteration.
            input_ids = np.append(input_ids, [[next_token_id]], axis=1)

        # Print a newline when the model finishes its complete thought
        print("\n")

In [67]:
def train_and_evaluate(num_epochs: int = 1000, eval_every_n_steps: int = 5, save_every_n_steps: int = 5):
    train_dataset: Dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    val_dataset: Dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")

    gpt2tokenizer: PreTrainedTokenizer = AutoTokenizer.from_pretrained("gpt2")
    gpt2tokenizer.pad_token = gpt2tokenizer.eos_token

    train_loader: grain.DataLoader = build_dataloader(
        train_dataset, gpt2tokenizer, batch_size=8, max_length=128
    )
    val_loader: grain.DataLoader = build_dataloader(
        val_dataset, gpt2tokenizer, batch_size=8, max_length=128
    )

    # Initialize Model and Optimizer

    rngs: nnx.Rngs = nnx.Rngs(0)
    model: nnx.Module = GatedDeltaNet(
        vocab_size=gpt2tokenizer.vocab_size,
        num_layers=6,
        dim=512,
        num_heads=8,
        chunk_size=16,
        rngs=rngs
    )

    """ GPTModel(
        vocab_size=gpt2tokenizer.vocab_size,
        embed_dim=512,
        num_q_heads=8,
        num_kv_heads=4,
        head_dim=64,
        seq_length=128,
        dropout_rate=0.1,
        n_layers=6,
        emb_dim_multiply=4,
        rngs=rngs,
    ) """
    optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=3e-4), wrt=nnx.Param)

    # 1. Define the directory path
    CHECKPOINT_DIR = ''

    """try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = os.path.abspath("/content/drive/My Drive/nugie_llm_checkpoints")
    except ImportError:
        CHECKPOINT_DIR = os.path.abspath("./nugie_llm_checkpoints")"""

    # 2. Configure the manager's behavior
    # options = ocp.CheckpointManagerOptions(max_to_keep=1, create=True)

    # 3. Initialize the 'mngr' variable
    # mngr = ocp.CheckpointManager(CHECKPOINT_DIR, options=options)

    data_iterator = iter(train_loader)

    step = 0 # restore_checkpoint(mngr, model, optimizer, rngs, data_iterator)

    print("Starting training...")
    if step > 0:
        print(f"Resuming training from step {step}...")

    for epoch in range(num_epochs):
        for batch in data_iterator:
            train_loss = train_step(model, optimizer, batch)

            if step % eval_every_n_steps == 0 and step > 0:
                total_val_loss = 0.0
                val_steps = 0

                for val_batch in val_loader:
                    val_loss = eval_step(model, val_batch)
                    total_val_loss += val_loss
                    val_steps += 1

                avg_val_loss = total_val_loss / val_steps
                perplexity = math.exp(avg_val_loss)

                print(
                    f"Val Loss: {avg_val_loss:.4f} | Perplexity: {perplexity:.2f} | Epoch: {epoch + 1}/{num_epochs}"
                )

            print(
                f"Step {step:04d} | Train Loss: {train_loss:.4f} | Epoch: {epoch + 1}/{num_epochs}"
            )
            step += 1

            # if step % save_every_n_steps == 0:
            #     save_checkpoint(mngr, step, model, optimizer, rngs, data_iterator)

        step = 0  # Reset step count after each epoch

    interactive_chat(model, gpt2tokenizer, max_new_tokens=150)

In [ ]:
import sys

from absl import flags

# flags.FLAGS(sys.argv)  # Parse flags to keep Grain multiprocessing happy

train_and_evaluate()

Starting training...
Step 0000 | Train Loss: 11.2063 | Epoch: 1/1000
Step 0001 | Train Loss: 10.7524 | Epoch: 1/1000
Step 0002 | Train Loss: 9.9590 | Epoch: 1/1000
Step 0003 | Train Loss: 6.1012 | Epoch: 1/1000
Step 0004 | Train Loss: 5.7682 | Epoch: 1/1000
